In [ ]:
all_indexes=get_all_securities(['index'])

In [ ]:
all_indexes[all_indexes.display_name.str.contains("中证全指")]

In [ ]:
# ============================================================
# cell 0: imports + 全局参数 + rolling_rank_pct helper
# ============================================================
from jqdata import *            # 聚宽 magic：finance / bond / get_index_valuation / get_price
import sys, os, gc
from pathlib import Path
from datetime import datetime, date

import pandas as pd
import numpy as np

# tqdm 在 Jupyter/聚宽研究环境里要用 notebook 版本才会渲染进度条 widget；
# 老环境 fallback 到 console 版本
try:
    from tqdm.notebook import tqdm
except ImportError:
    try:
        from tqdm.auto import tqdm
    except ImportError:
        from tqdm import tqdm

# 项目根路径（聚宽研究环境当前目录）
PROJ = Path('./')
sys.path.insert(0, str(PROJ.resolve()))

# 全局参数
AS_OF             = date.today()
LOOKBACK_DAYS     = 5 * 252 + 60     # 5 年 + 60 交易日 buffer
QUANTILE_WINDOW   = 1260             # 5 年分位窗口（与项目全局一致）
SHORT_WINDOW      = 252              # 1 年短期窗口
INDEX_CODE        = '000985.XSHG'    # 中证全指（PE 口径）
ERP_PERIOD_WINDOW = 252              # ERP 趋势线滚动窗口（1 年）

# 自定义 rolling rank pct 工具（pandas 1.1.5 不支持 .rolling().rank）
def rolling_rank_pct(series, window, min_periods=252):
    '计算滚动窗口内末位值的 rank pct；样本不足返回 NaN。'
    def _rank_last(x):
        if len(x) < min_periods:
            return np.nan
        return x.rank(pct=True).iloc[-1]
    return series.rolling(window=window, min_periods=min_periods).apply(_rank_last)



# 06 · 资金与风险偏好面板

实现《系统性风险看板.md》"模块五：资金与风险偏好面板"的三个核心指标：

1. **两融余额及增速** —— 杠杆资金情绪
2. **北向资金 20/60 日累计净流入** —— 外资偏好
3. **股权风险溢价 ERP** —— 股票/债券性价比中枢

## 数据源

| 指标 | 聚宽 API | 字段 | 状态 |
|------|---------|------|------|
| 两融余额 | `finance.STK_MT_TOTAL`（按 exchange_code 聚合 `fin_sec_value`） | 沪深两市融资融券合计，元 | ✓ |
| 北向资金 | `finance.STK_HK_HOLD_INFO`（link_id ∈ {310001, 310002}） | `share_number` diff × 收盘价 | ✓ **（2024-08-17 起按季度披露）** |
| ERP（股票端） | `get_fundamentals` + `valuation.pe_ratio`（中证全指成分股 25-75 分位 PE-TTM 中位数） | 中位数 PE-TTM | ✓ |
| ERP（债券端） | `bond.REPO_DAILY_PRICE` GC001 + 60 日滚动均值 | `close` 短端利率代理平滑后，% | ✓ |

## 风险提示

1. **REPO_DAILY_PRICE 不是真正的 10 年国债收益率**，仅作为短端资金面松紧代理——聚宽研究环境缺少中长期国债收益率接口（本方案加 60 日滚动均值平滑后再用作 ERP 计算）。
2. ERP 与中金/万得官方 ERP 口径有差异（"中证全指 PE-TTM 中位数 + 逆回购 60 日均值" vs "沪深 300 EP − 10Y 国债"），数值不可直接对比，仅供同源历史分位参考。
3. 北向资金历史起点：沪股通 2014-11、深股通 2016-12，启动前 `STK_HK_HOLD_INFO` 为空。
4. 5 年分位窗口起步 ~1260 个交易日为 NaN，是设计预期（rolling 避免 look-ahead bias）。
5. 本模块只输出局部档位（-1/0/1）与模块五综合得分；最终的"舒适/审慎/防御/危机"四档综合仓位建议在主看板汇总所有六个模块后输出。
6. **本 notebook 不落盘**：所有图用 `plt.show()` 直接显示，不写 CSV / PNG。

## 兼容性

- Python 3.6.7：无 PEP 604 union、无 dataclass、无 f"{x=}"
- pandas 1.1.5：`resample` 用 `"M"` 不用 `"ME"`；**`Rolling.rank()` 不存在**，分位用 cell 0 的 `rolling_rank_pct()`
- matplotlib 老版本：`ax.plot(x.to_pydatetime(), series.values)`
- `finance.run_query` 单次 ≤ 4000 行；北向按天分页循环 + 流式算 net_inflow（避免 1G 内存超限）
- `tqdm.notebook` 在聚宽 Jupyter 中渲染 widget 进度条；老环境 fallback 到 console 版



In [ ]:
# ============================================================
# cell 2: 拉两融余额（沪深两市）原始数据
# ============================================================
# finance.STK_MT_TOTAL 字段：date, exchange_code (XSHG/XSHE), fin_value, fin_sec_value
# fin_sec_value = 融资融券余额合计（元）
start_date = AS_OF - pd.Timedelta(days=int(LOOKBACK_DAYS * 1.5))   # 预留 buffer

q = query(finance.STK_MT_TOTAL).filter(
    finance.STK_MT_TOTAL.date >= start_date,
    finance.STK_MT_TOTAL.exchange_code.in_(['XSHG', 'XSHE'])
)
df_margin_raw = finance.run_query(q)
print(f"两融原始记录数：{len(df_margin_raw):,}")
print(f"日期范围：{df_margin_raw['date'].min()} ~ {df_margin_raw['date'].max()}")
print(f"交易所分布：\n{df_margin_raw['exchange_code'].value_counts()}")
df_margin_raw.head()



In [ ]:
# ============================================================
# cell 3: 聚合每日沪深两融合计 + 同比/环比 + 5 年分位
# ============================================================
df_margin_total = (
    df_margin_raw
    .groupby('date', as_index=False)['fin_sec_value']
    .sum()
    .sort_values('date')
    .set_index('date')
)
df_margin_total.index = pd.to_datetime(df_margin_total.index)
df_margin_total.columns = ['两融余额']

# 同比（250 个交易日 ≈ 1 年）、环比（20 个交易日 ≈ 1 月）
df_margin_total['两融_同比增速'] = df_margin_total['两融余额'].pct_change(250)
df_margin_total['两融_环比增速'] = df_margin_total['两融余额'].pct_change(20)

# 5 年分位（rolling_rank_pct 避免 look-ahead bias + 兼容 pandas 1.1.5）
df_margin_total['两融_5年分位'] = rolling_rank_pct(
    df_margin_total['两融余额'],
    window=QUANTILE_WINDOW,
    min_periods=252,
)
print(f"两融汇总 shape：{df_margin_total.shape}")
_first_valid = df_margin_total["两融_5年分位"].dropna()
_start = _first_valid.index.min().date() if len(_first_valid) > 0 else "N/A"
print(f"有效分位起始：{_start}")
df_margin_total.tail()



In [ ]:
# ============================================================
# cell 4: 北向资金流式按日拉取 + 当日算 net_inflow（增量更新，避免 1G 内存超限）
# ============================================================
# finance.STK_HK_HOLD_INFO 字段：day, link_id (310001=沪股通 310002=深股通),
#                                 code, share_number, share_ratio
#
# 增量更新策略：
#   历史 df_north_daily 持久化到 NORTH_DAILY_CSV；本次只算 history_max_date+1 ~ today。
#   增量首日（history_max_date+1）的 df_prev 需要从 history_max_date 的快照重建：
#     额外拉一次 history_max_date 的快照作为初始 df_prev，保证 diff 正确衔接。
#
# 内存策略：
#   聚宽研究环境最多 1G 内存。北向资金 5 年累计 ~190 万行（全量拉取会爆）。
#   改为按天循环：每天拉一次快照 (~1500 行) → 与昨日快照做 diff → 立即算当日
#   net_inflow 累加 → 释放当日内存。峰值内存 < 1 MB（仅 df_prev + 累加 list）。
NORTH_DAILY_CSV = PROJ / 'outputs' / 'tmp_north_daily.csv'

# Step 1: 加载历史（如有）
df_history = None
history_max_date = None
if NORTH_DAILY_CSV.exists():
    df_history = pd.read_csv(NORTH_DAILY_CSV, parse_dates=['day'], index_col='day').sort_index()
    df_history.columns = ['北向资金日度净流入']
    history_max_date = df_history.index[-1].date()
    print(f"已加载历史 {len(df_history):,} 行（最近日期：{history_max_date}，累计净流入 {df_history['北向资金日度净流入'].sum()/1e8:+.1f} 亿元）")

# Step 2: 确定增量计算起点
if history_max_date is not None:
    calc_start_date = history_max_date + pd.Timedelta(days=1)
else:
    calc_start_date = start_date
new_days = get_trade_days(start_date=calc_start_date.strftime("%Y-%m-%d"),
                          end_date=AS_OF.strftime("%Y-%m-%d"))
print(f"增量计算范围：{calc_start_date} ~ {AS_OF}（{len(new_days):,} 个交易日）")

# Step 3: 重建 df_prev（仅增量模式需要）
df_prev = None
if history_max_date is not None:
    # 拉 history_max_date 当天的快照作为初始 df_prev
    q_prev = query(finance.STK_HK_HOLD_INFO).filter(
        finance.STK_HK_HOLD_INFO.day == history_max_date,
        finance.STK_HK_HOLD_INFO.link_id.in_([310001, 310002]),
    )
    df_prev = finance.run_query(q_prev)[['code', 'share_number']].copy()
    print(f"已加载 {history_max_date} 快照作为初始 df_prev（{len(df_prev):,} 只股）")

# Step 4: 流式增量循环
north_net = []                                              # 仅累加新增日
pbar = tqdm(new_days, desc='北向按日', unit='day')
for day in pbar:
    pbar.set_postfix(新增天数=len(north_net))
    q = query(finance.STK_HK_HOLD_INFO).filter(
        finance.STK_HK_HOLD_INFO.day == day,
        finance.STK_HK_HOLD_INFO.link_id.in_([310001, 310002]),
    )
    chunk = finance.run_query(q)
    if len(chunk) == 0:
        continue
    chunk['day'] = pd.to_datetime(chunk['day'])

    # 与昨日快照做 diff（首次出现 fillna(今日值) → delta=0，与首次出现 dropna 行为一致）
    if df_prev is not None and len(df_prev) > 0:
        merged = chunk.merge(df_prev, on='code', how='left', suffixes=('', '_prev'))
        merged['share_number_prev'] = merged['share_number_prev'].fillna(merged['share_number'])
        merged['delta_shares'] = merged['share_number'] - merged['share_number_prev']

        # 取当日收盘价（只查涉及的 ~1500 只股一天的数据，开销小）
        codes_today = merged['code'].unique().tolist()
        price_today = get_price(
            codes_today,
            start_date=str(day), end_date=str(day),
            frequency='daily', fields=['close'],
            panel=False, skip_paused=False, fq='pre', fill_paused=True,
        )
        price_today = price_today.rename(columns={'time': 'day'})
        price_today['day'] = pd.to_datetime(price_today['day'])

        merged = merged.merge(price_today, on=['code', 'day'], how='left')
        merged['net_inflow'] = merged['delta_shares'] * merged['close']
        daily_inflow = merged['net_inflow'].sum()
        north_net.append({'day': pd.Timestamp(day), '北向资金日度净流入': daily_inflow})

        del merged, price_today, codes_today
        gc.collect()

    # 今日变昨日
    df_prev = chunk[['code', 'share_number']].copy()
    del chunk
    gc.collect()
pbar.close()

# Step 5: 合并历史 + 增量
if north_net:
    df_new = pd.DataFrame(north_net).set_index('day').sort_index()
    if df_history is not None:
        df_north_daily = pd.concat([df_history, df_new]).sort_index()
        df_north_daily = df_north_daily[~df_north_daily.index.duplicated(keep='last')]
    else:
        df_north_daily = df_new
    print(f"本次新增 {len(df_new):,} 行，总计 {len(df_north_daily):,} 行")
else:
    df_north_daily = df_history if df_history is not None else pd.DataFrame(columns=['北向资金日度净流入'])
    print(f"无新增数据，沿用历史 ({len(df_north_daily):,} 行)")

# Step 6: 持久化到 CSV（中间缓存，不是最终输出）
NORTH_DAILY_CSV.parent.mkdir(parents=True, exist_ok=True)
df_north_daily.to_csv(NORTH_DAILY_CSV, encoding='utf-8-sig')
print(f"已保存到 {NORTH_DAILY_CSV}")
print(f"日期范围：{df_north_daily.index.min().date()} ~ {df_north_daily.index.max().date()}")
print(f"累计净流入（亿元）：{df_north_daily['北向资金日度净流入'].sum() / 1e8:+.1f}")
df_north_daily.tail()


In [ ]:
# ============================================================
# cell 5: 滚动累计 + 1 年分位
# ============================================================
# 20/60 日累计净流入（min_periods 取窗口一半，平衡起步段 NaN 与样本量）
df_north_daily['北向资金_20日累计'] = (
    df_north_daily['北向资金日度净流入'].rolling(window=20, min_periods=10).sum()
)
df_north_daily['北向资金_60日累计'] = (
    df_north_daily['北向资金日度净流入'].rolling(window=60, min_periods=30).sum()
)

# 1 年分位（针对累计口径；用 rolling_rank_pct 兼容 pandas 1.1.5）
df_north_daily['北向_20日累计_1年分位'] = rolling_rank_pct(
    df_north_daily['北向资金_20日累计'],
    window=SHORT_WINDOW,
    min_periods=60,
)
df_north_daily['北向_60日累计_1年分位'] = rolling_rank_pct(
    df_north_daily['北向资金_60日累计'],
    window=SHORT_WINDOW,
    min_periods=60,
)
df_north_daily.tail()



In [ ]:
# ============================================================
# cell 6: 中证全指 PE-TTM（成分股中位数算法 + 增量缓存）
# ============================================================
# 临时参考.md 算法：对每个交易日取指数成分股，25-75 分位过滤极端值后取
# PE-TTM 中位数。比 get_index_valuation（市值加权 PE）更稳健（不受单只大权重股影响）。
# 注意：聚宽研究环境没有 get_index_valuation，用 get_fundamentals + valuation.pe_ratio。
#
# 性能：5 年约 1260 个交易日 × 每次 get_fundamentals 查询 ~3000 只股 = 较慢（首次约 10-30 分钟）。
# 增量缓存：df_pe_pb 保存到 ./data/中证全指_pe_pb.csv，下次运行只算历史最大日期之后的增量。
PE_PB_CSV = PROJ / 'data' / '中证全指_pe_pb.csv'

# 加载本地缓存
df_pe_cached = pd.DataFrame()
if PE_PB_CSV.exists():
    df_pe_cached = pd.read_csv(PE_PB_CSV, parse_dates=['date'], index_col='date').sort_index()
    print(f"已加载本地缓存 {len(df_pe_cached):,} 行（最近日期：{df_pe_cached.index[-1].date()}）")

# 确定增量计算起点
if len(df_pe_cached) > 0:
    calc_start_date = df_pe_cached.index[-1].date() + pd.Timedelta(days=1)
else:
    calc_start_date = start_date
print(f"增量计算范围：{calc_start_date} ~ {AS_OF}")

# 流式增量拉取
new_pe_pb = []
all_days = get_trade_days(start_date=calc_start_date.strftime("%Y-%m-%d"),
                          end_date=AS_OF.strftime("%Y-%m-%d"))
pbar = tqdm(all_days, desc='中证全指 PE/PB 中位数', unit='day')
for date in pbar:
    stocks = get_index_stocks(INDEX_CODE, date)
    if not stocks:
        continue
    q = query(valuation.pe_ratio, valuation.pb_ratio).filter(
        valuation.pe_ratio != None,
        valuation.pb_ratio != None,
        valuation.code.in_(stocks)
    )
    df = get_fundamentals(q, date)
    if df is None or len(df) == 0:
        continue
    # 25-75 分位过滤极端值
    quantile = df.quantile([0.25, 0.75])
    df_pe_valid = df['pe_ratio'][
        (df['pe_ratio'] > quantile['pe_ratio'].values[0]) &
        (df['pe_ratio'] < quantile['pe_ratio'].values[1])
    ]
    df_pb_valid = df['pb_ratio'][
        (df['pb_ratio'] > quantile['pb_ratio'].values[0]) &
        (df['pb_ratio'] < quantile['pb_ratio'].values[1])
    ]
    new_pe_pb.append({
        'date': pd.Timestamp(date),
        'pe': df_pe_valid.median(),
        'pb': df_pb_valid.median(),
    })
pbar.close()

# 合并 + 保存
if new_pe_pb:
    df_new = pd.DataFrame(new_pe_pb).set_index('date').sort_index()
    if len(df_pe_cached) > 0:
        df_pe_pb = pd.concat([df_pe_cached, df_new]).sort_index()
        df_pe_pb = df_pe_pb[~df_pe_pb.index.duplicated(keep='last')]
    else:
        df_pe_pb = df_new
    PE_PB_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_pe_pb.to_csv(PE_PB_CSV, encoding='utf-8-sig')
    print(f"本次新增 {len(df_new):,} 行，总计 {len(df_pe_pb):,} 行")
else:
    df_pe_pb = df_pe_cached
    print(f"无新增数据，沿用本地缓存 ({len(df_pe_pb):,} 行)")

# 提取 PE-TTM 列给后续 cell 用
df_pe = df_pe_pb[['pe']].copy()
df_pe.columns = ['中证全指_PE_TTM']
print(f"PE 中位数范围：{df_pe['中证全指_PE_TTM'].min():.2f} ~ {df_pe['中证全指_PE_TTM'].max():.2f}")
df_pe.head()



In [ ]:
df_pe.tail()

In [ ]:
# ============================================================
# cell 7: 国债逆回购利率（GC001）作为 10 年国债收益率代理（60 日平滑）
# ============================================================
# bond.REPO_DAILY_PRICE 字段：date, code, name, exchange_code,
#   pre_close/open/high/low/close（利率 %）, volume（手）, money（元）, deal_number
#
# 用 name 字段过滤 GC001（上海 1 天期国债逆回购，流动性最好）。
# 注：GC001 是短端利率代理（真实 10 年国债在 2.5-3.5% 区间窄幅波动），
#     GC001 日度波动较大（1.45-5.39%），故加 60 日滚动均值平滑后再用作 ERP 计算。
#
# ⚠️ 重要 bug 修复：聚宽研究环境下 bond.REPO_DAILY_PRICE 的 date filter 不下推到数据库，
#    加上 GC001 全量 4882 行超过 run_query 单次 4000 行上限 → 必须先按 name 拉全量再内存 filter date。
df_repo = bond.run_query(query(bond.REPO_DAILY_PRICE).filter(
    bond.REPO_DAILY_PRICE.name == 'GC001',
))
df_repo['date'] = pd.to_datetime(df_repo['date'])
# 内存中按 start_date 过滤（GC001 全量 4882 行，未超 4000 上限）
df_repo = df_repo[df_repo['date'] >= pd.Timestamp(start_date)].copy()
df_repo = df_repo.set_index('date').sort_index()[['close']]
df_repo.columns = ['国债逆回购利率_日度']
# 60 日滚动均值：消除日度周期性波动，更接近 10 年国债实际水平
df_repo['国债逆回购利率_60日均值'] = df_repo['国债逆回购利率_日度'].rolling(window=60, min_periods=20).mean()
print(f"国债逆回购代理 shape：{df_repo.shape}")
print(f"日期范围：{df_repo.index.min().date()} ~ {df_repo.index.max().date()}")
print(f"日度利率范围：{df_repo['国债逆回购利率_日度'].min():.3f}% ~ {df_repo['国债逆回购利率_日度'].max():.3f}%")
print(f"60 日均值范围：{df_repo['国债逆回购利率_60日均值'].min():.3f}% ~ {df_repo['国债逆回购利率_60日均值'].max():.3f}%")
df_repo.head()



In [ ]:
# ============================================================
# cell 8: ERP 计算（用 60 日均值利率代理）+ 1 年趋势 + 5 年分位
# ============================================================
# ERP = 1/PE − 利率/100：PE 是倍数（如 18.0）、利率是 %（如 2.5），输出为小数
# 利率端用 GC001 60 日均值（消除日度周期波动），更接近实际长期利率水平
df_erp = df_pe.join(df_repo, how='inner')
df_erp['ERP'] = 1.0 / df_erp['中证全指_PE_TTM'] - df_erp['国债逆回购利率_60日均值'] / 100.0

# 1 年滚动均值（ERP 自身的趋势线）
df_erp['ERP_1年均值'] = df_erp['ERP'].rolling(window=ERP_PERIOD_WINDOW, min_periods=60).mean()

# 5 年分位（rolling_rank_pct 兼容 pandas 1.1.5）
df_erp['ERP_5年分位'] = rolling_rank_pct(
    df_erp['ERP'],
    window=QUANTILE_WINDOW,
    min_periods=252,
)
print(f"ERP 序列 shape：{df_erp.shape}")
print(f"ERP 范围：{df_erp['ERP'].min():.2%} ~ {df_erp['ERP'].max():.2%}")
print(f"ERP 1 年均值范围：{df_erp['ERP_1年均值'].min():.2%} ~ {df_erp['ERP_1年均值'].max():.2%}")
df_erp.tail()



In [ ]:
df_erp['ERP'].plot()

In [ ]:
# ============================================================
# cell 9: 三大模块横向合并（两融 + 北向 + ERP，按日期 outer join）
# ============================================================
# ⚠️ 2024-08 之后北向按季度披露：北向 310001/310002 自 2024-08-17 起改为按季度披露。
#   因此用 outer join：保留两融和 ERP 的全部日度日期（~1300 行），
#   北向列在 2024-08 后大部分日期是 NaN、只有 8 个季度末日期有值。
#   这样后续图表能呈现：
#     - 两融/ERP：连续日度曲线
#     - 北向：2024-08 前连续 + 2024-08 后断线 + 8 个季度末点
df_module5 = (
    df_margin_total[['两融余额', '两融_5年分位']]
    .join(df_north_daily[['北向资金_20日累计', '北向资金_60日累计',
                           '北向_20日累计_1年分位', '北向_60日累计_1年分位']], how='left')
    .join(df_erp[['ERP', 'ERP_1年均值', 'ERP_5年分位']], how='left')
)
print(f"模块五合并后 shape：{df_module5.shape}")
print(f"日期范围：{df_module5.index.min().date()} ~ {df_module5.index.max().date()}")
print(f"列：{list(df_module5.columns)}")
print(f"两融有效行：{df_module5['两融余额'].notna().sum()} / {len(df_module5)}")
print(f"北向有效行：{df_module5['北向资金_20日累计'].notna().sum()} / {len(df_module5)}  ← 2024-08 后为季度末 8 个点")
print(f"ERP有效行：{df_module5['ERP'].notna().sum()} / {len(df_module5)}")
df_module5.tail()



In [ ]:
# ============================================================
# cell 10: 单模块局部平滑（连续 2 日触发阈值才切换）
# ============================================================
# 三个指标的阈值档位定义
THRESHOLDS = {
    '两融_5年分位':         {'high': 0.80, 'low': 0.20},   # 高=加杠杆热(风险) 低=去杠杆(机会)
    '北向_20日累计_1年分位': {'high': 0.80, 'low': 0.20},   # 高=外资加速流入(机会) 低=外资撤(风险)
    'ERP_5年分位':          {'high': 0.80, 'low': 0.20},   # 高=股债性价比高(机会) 低=股票过热(风险)
}

def smooth_signal(series, high, low, min_consec=2):
    '连续 min_consec 日满足阈值才切换档位（-1/0/1），否则沿用昨日档位。'
    raw = pd.Series(np.nan, index=series.index)
    raw[series > high] = 1     # 高位
    raw[series < low]  = -1    # 低位
    raw[(series >= low) & (series <= high)] = 0  # 中性
    smoothed = raw.copy()
    for i in range(min_consec, len(raw)):
        window = raw.iloc[i - min_consec + 1: i + 1]
        if window.notna().all() and window.nunique() == 1:
            smoothed.iloc[i] = window.iloc[-1]
        else:
            smoothed.iloc[i] = smoothed.iloc[i - 1]   # 沿用昨日档位
    return smoothed

df_signals = pd.DataFrame(index=df_module5.index)
for col, thr in THRESHOLDS.items():
    df_signals[col + '_信号'] = smooth_signal(df_module5[col], thr['high'], thr['low'])
print("平滑信号表（前 5 行）：")
print(df_signals.head())
print("\n平滑信号表（最新 5 行）：")
print(df_signals.tail())



In [ ]:
# ============================================================
# cell 11: 最新一期指标 + 模块五综合档位
# ============================================================
latest = df_module5.iloc[-1]
print("=" * 60)
print("模块五【资金与风险偏好面板】最新读数")
print("=" * 60)
print(f"日期：{df_module5.index[-1].date()}")
print(f"两融余额（亿元）：{latest['两融余额']/1e8:,.0f}, 5年分位：{latest['两融_5年分位']:.1%}")
print(f"北向20日累计（亿元）：{latest['北向资金_20日累计']/1e8:+,.1f}, 1年分位：{latest['北向_20日累计_1年分位']:.1%}")
print(f"北向60日累计（亿元）：{latest['北向资金_60日累计']/1e8:+,.1f}, 1年分位：{latest['北向_60日累计_1年分位']:.1%}")
print(f"ERP：{latest['ERP']:.2%}, 1年均值：{latest['ERP_1年均值']:.2%}, 5年分位：{latest['ERP_5年分位']:.1%}")

# 模块五综合得分（0-100，三指标各 1/3 权重）：
# 两融 高=风险(1) 中=中性(0.5) 低=机会(0)
# 北向 高=机会(0) 中=中性(0.5) 低=风险(1)
# ERP  高=机会(0) 中=中性(0.5) 低=风险(1)
def risk_contrib(row):
    """NaN 容错：任一信号为 NaN（如分位起步段、2024-08 后北向季度披露）返回 NaN。"""
    s_mz = row['两融_5年分位_信号']
    s_nb = row['北向_20日累计_1年分位_信号']
    s_er = row['ERP_5年分位_信号']
    if pd.isna(s_mz) or pd.isna(s_nb) or pd.isna(s_er):
        return np.nan
    mz = {1: 1.0, 0: 0.5, -1: 0.0}[s_mz]
    nb = {1: 0.0, 0: 0.5, -1: 1.0}[s_nb]
    er = {1: 0.0, 0: 0.5, -1: 1.0}[s_er]
    return (mz + nb + er) / 3.0 * 100

risk_score = df_signals.apply(risk_contrib, axis=1)
print("\n模块五历史综合得分（最新 5 行）：")
print(risk_score.tail())
print(f"\n模块五最新综合得分（0=机会 100=风险）：{risk_score.iloc[-1]:.1f}")
print(f"模块五 60 日均值：{risk_score.rolling(60).mean().iloc[-1]:.1f}")



In [ ]:
# ============================================================
# cell 12: 图1 - 两融余额时序 + 5 年分位（双 y 轴）
# ============================================================
x_vals = df_module5.index.to_pydatetime()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(x_vals, df_module5['两融余额'].values / 1e8,
        color='#1f77b4', lw=1.5, label='两融余额(亿元)')
ax.set_ylabel('两融余额（亿元）', color='#1f77b4', fontsize=12)
ax.set_xlabel('日期', fontsize=12)
ax.set_title('两融余额 + 5 年分位｜高=杠杆热(风险) / 低=去杠杆(机会)', fontsize=14)
ax.grid(alpha=0.3)
ax.legend(loc='upper left')
# 标注 2024-08-19 起北向按季度披露（背景标注，不影响两融数据本身）
ax.axvspan(pd.Timestamp('2024-08-19'), df_module5.index[-1],
            alpha=0.10, color='gray')

ax2 = ax.twinx()
ax2.plot(x_vals, df_module5['两融_5年分位'].values,
         color='black', lw=1.0, linestyle='--', alpha=0.7, label='5 年分位')
ax2.axhline(0.80, color='red', lw=0.8, alpha=0.5, linestyle='--')
ax2.axhline(0.20, color='green', lw=0.8, alpha=0.5, linestyle='--')
ax2.set_ylabel('5 年分位（0-1）', color='black', fontsize=12)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# cell 13: 图2 - 北向资金 20/60 日累计 + 1 年分位
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle('北向资金 20/60 日累计净流入（亿元）', fontsize=16, y=0.96)

# 20 日累计
ax1 = axes[0]
ax1.plot(x_vals, df_module5['北向资金_20日累计'].values / 1e8,
         color='#1f77b4', lw=1.5, label='20 日累计(亿元)')
ax1.axhline(0, color='black', lw=0.5)
ax1.set_title('北向资金 20 日累计净流入', fontsize=13)
ax1.set_ylabel('亿元', fontsize=11)
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(x_vals, df_module5['北向_20日累计_1年分位'].values,
         color='black', lw=1.0, linestyle='--', alpha=0.6,
         marker='o', markersize=6, markerfacecolor='red',
         markeredgecolor='darkred',
         label='1 年分位（2024-08 后为季度末 8 个点）')
# 2024-08 后手动连一条虚线（matplotlib 默认在 NaN 处断开）
_north_after_2024 = df_module5.loc['2024-08-19':, '北向_20日累计_1年分位'].dropna()
if len(_north_after_2024) > 0:
    ax2.plot(_north_after_2024.index.to_pydatetime(), _north_after_2024.values,
             color='black', lw=1.2, linestyle='--', alpha=0.7, zorder=4)
ax2.axhline(0.80, color='red', lw=0.8, alpha=0.5, linestyle='--')
ax2.axhline(0.20, color='green', lw=0.8, alpha=0.5, linestyle='--')
ax2.set_ylabel('1 年分位', color='black', fontsize=11)
ax2.set_ylim(0, 1)
# 标注 2024-08-19 起按季度披露区域
ax2.axvspan(pd.Timestamp('2024-08-19'), df_module5.index[-1],
            alpha=0.15, color='gray')
ax2.legend(loc='upper right')

# 60 日累计
ax3 = axes[1]
ax3.plot(x_vals, df_module5['北向资金_60日累计'].values / 1e8,
         color='#ff7f0e', lw=1.5, label='60 日累计(亿元)')
ax3.axhline(0, color='black', lw=0.5)
ax3.set_title('北向资金 60 日累计净流入', fontsize=13)
ax3.set_ylabel('亿元', fontsize=11)
ax3.set_xlabel('日期', fontsize=11)
ax3.legend(loc='upper left')
ax3.grid(alpha=0.3)
# 标注 2024-08-19 起按季度披露区域
ax3.axvspan(pd.Timestamp('2024-08-19'), df_module5.index[-1],
            alpha=0.15, color='gray')

ax4 = ax3.twinx()
ax4.plot(x_vals, df_module5['北向_60日累计_1年分位'].values,
         color='black', lw=1.0, linestyle='--', alpha=0.6,
         marker='o', markersize=6, markerfacecolor='red',
         markeredgecolor='darkred', label='1 年分位（2024-08 后为季度末 8 个点）')
# 2024-08 后手动连一条虚线
_north_after_2024 = df_module5.loc['2024-08-19':, '北向_60日累计_1年分位'].dropna()
if len(_north_after_2024) > 0:
    ax4.plot(_north_after_2024.index.to_pydatetime(), _north_after_2024.values,
             color='black', lw=1.2, linestyle='--', alpha=0.7, zorder=4)
ax4.axhline(0.80, color='red', lw=0.8, alpha=0.5, linestyle='--')
ax4.axhline(0.20, color='green', lw=0.8, alpha=0.5, linestyle='--')
ax4.set_ylabel('1 年分位', color='black', fontsize=11)
ax4.set_ylim(0, 1)
ax4.legend(loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# cell 14: 图3 - ERP 时序 + 1 年趋势 + 5 年分位（双 y 轴）
# ============================================================
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(x_vals, df_module5['ERP'].values * 100,
        color='#1f77b4', lw=1.0, alpha=0.6, label='ERP 日度(%)')
ax.plot(x_vals, df_module5['ERP_1年均值'].values * 100,
        color='#d62728', lw=2.0, label='ERP 1 年均值(%)')
ax.axhline(0, color='black', lw=0.5)
ax.set_title('股权风险溢价 ERP（下行=股票过热 / 上行=股票便宜）', fontsize=14)
ax.set_ylabel('ERP (%)', fontsize=12)
ax.set_xlabel('日期', fontsize=12)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
# 标注 2024-08-19 起北向按季度披露（背景标注，不影响 ERP 数据本身）
ax.axvspan(pd.Timestamp('2024-08-19'), df_module5.index[-1],
            alpha=0.10, color='gray')

ax2 = ax.twinx()
ax2.plot(x_vals, df_module5['ERP_5年分位'].values,
         color='black', lw=1.0, linestyle='--', alpha=0.7, label='5 年分位')
ax2.axhline(0.80, color='green', lw=0.8, alpha=0.5, linestyle='--',
            label='高分位=机会')
ax2.axhline(0.20, color='red', lw=0.8, alpha=0.5, linestyle='--',
            label='低分位=风险')
ax2.set_ylabel('5 年分位（0-1）', color='black', fontsize=12)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# cell 15: 图4 - 三大核心指标分位时序对比（统一 0-1 y 轴）
# ============================================================
fig, ax = plt.subplots(figsize=(16, 7))
# 两融：日度连续
ax.plot(x_vals, df_module5['两融_5年分位'].values,
        color='#1f77b4', lw=1.5, label='两融 5 年分位（高=风险）')
# 北向：2024-08 后按季度披露，加红色散点 + 灰色背景标注
ax.plot(x_vals, df_module5['北向_60日累计_1年分位'].values,
        color='#ff7f0e', lw=1.5,
        marker='o', markersize=6, markerfacecolor='red',
        markeredgecolor='darkred',
        label='北向 60 日累计 1 年分位（低=风险，2024-08 后季度末点）')
# 2024-08 后手动连一条虚线（matplotlib 默认 NaN 处断开）
_north_after_2024 = df_module5.loc['2024-08-19':, '北向_60日累计_1年分位'].dropna()
if len(_north_after_2024) > 0:
    ax.plot(_north_after_2024.index.to_pydatetime(), _north_after_2024.values,
            color='#ff7f0e', lw=1.7, linestyle='-', alpha=0.8, zorder=4)
# ERP：日度连续
ax.plot(x_vals, df_module5['ERP_5年分位'].values,
        color='#2ca02c', lw=1.5, label='ERP 5 年分位（低=风险）')
ax.axhline(0.80, color='red', lw=0.8, alpha=0.4, linestyle='--', label='高分位阈值 0.80')
ax.axhline(0.20, color='green', lw=0.8, alpha=0.4, linestyle='--', label='低分位阈值 0.20')
ax.axhline(0.50, color='gray', lw=0.5, alpha=0.3)
# 标注 2024-08-19 起北向按季度披露区域
ax.axvspan(pd.Timestamp('2024-08-19'), df_module5.index[-1],
            alpha=0.15, color='gray')
ax.set_title('模块五｜三大核心指标分位时序对比', fontsize=14)
ax.set_ylabel('历史分位（0-1）', fontsize=12)
ax.set_xlabel('日期', fontsize=12)
ax.set_ylim(0, 1)
ax.legend(loc='upper left', ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 模块五结论

本 notebook 实现《系统性风险看板.md》"模块五：资金与风险偏好面板"的三大核心指标，**只在 ipynb 内显示，不落盘**。

**核心读数（最新交易日，运行时由 cell 11 输出）**：
- 两融余额（亿元）+ 5 年分位 → 杠杆资金热度
- 北向 60 日累计净流入（亿元）+ 1 年分位 → 外资偏好
- ERP（%）+ 1 年均值 + 5 年分位 → 股债性价比（用 PE-TTM 中位数 + GC001 60 日均值）

**风控含义**：
- 三指标分位 **> 0.80**（同时）：资金情绪过热、估值泡沫化，建议降低权益仓位
- 三指标分位 **< 0.20**（同时）：资金撤出 / 估值底部，权益资产性价比凸显，可适度加仓
- 单指标极值不触发动作，需多指标共振（参考《系统性风险看板.md》四层档位）

## 已知数据 / 口径限制

1. **ERP 代理偏差（两层）**：
   - **股票端**：用中证全指成分股 25-75 分位 PE-TTM 中位数（比市值加权 PE 更稳健）。聚宽研究环境没有 `get_index_valuation`，改用 `get_fundamentals` + `valuation.pe_ratio` 自组装。
   - **债券端**：`bond.REPO_DAILY_PRICE` 是 1 天期国债逆回购利率 GC001，**不等于 10 年国债到期收益率**。聚宽研究环境**没有 10 年国债收益率、国债期货、SHIBOR、DR007 等中长期利率 API**（已查证）。本方案加 60 日滚动均值消除日度周期波动，更接近长期利率水平，但绝对值仍不准。如需更严谨的 ERP，建议自行准备 10 年国债收益率 CSV（如来自 Wind / 中债登）替换 cell 7 的数据源。

2. **北向资金历史完整性**：
   - 沪股通 2014-11-17 启动
   - 深股通 2016-12-05 启动
   - 启动前的 `STK_HK_HOLD_INFO` 记录为空，本模块历史有效起点为 2016-12 之后
   - **2024-08-17 起按季度披露**（沪股通/深股通），之前是日度
   - 流式按天处理：cell 4 中第一天（前一日无数据）的净流入无法计算，从第二天开始有值
   - 下次更新预计 2026-10 月初（盘前 6:30）披露 2026-09-30 数据

3. **两融余额历史完整性**：融资融券业务 2010-03-31 启动；早期数据可能存在交易所差异，建议只关注 2014 年之后。

4. **5 年分位起步段 NaN**：rolling window=1260，前 ~1260 个交易日的分位列为 NaN，是设计预期（避免 look-ahead bias）。

5. **不做跨模块综合档位**：本 notebook 只输出模块五的局部档位（-1/0/1）与综合得分（0-100），最终的"舒适/审慎/防御/危机"四档综合仓位建议需在主看板 notebook 中汇总所有六个模块后输出（模块五权重 10%，见《系统性风险看板.md》第三节）。

6. **不落盘约束**：所有图表用 `plt.show()` 直接显示，不写 CSV / PNG。

7. **内存策略**：cell 4 北向资金按天流式处理——每天拉一次快照 (~1500 行) → 立即与昨日 diff 算 net_inflow → 累加到 list → 释放当日内存。峰值 < 1 MB（远低于聚宽 1G 限制），5 年约 1260 次循环。

8. **增量更新与 CSV 中间缓存**：
   - cell 4：`df_north_daily` 持久化到 `outputs/tmp_north_daily.csv`，下次运行自动加载历史 CSV，只算增量
   - cell 6：PE/PB 持久化到 `data/中证全指_pe_pb.csv`，下次运行自动续接
   - 两个 CSV 都是中间缓存，不是最终输出。强制全量重算：删除对应 CSV 即可
